In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)



Minimum set Metrics:

Accuracy (for completeness)

Precision

Recall (Sensitivity)

F1-score

ROC-AUC

PR-AUC

If possible:

Specificity

Confusion matrix

### Step 0 : Dataset 

In [2]:
# --- your paths (same as notebook) ---
SRC_STREMLINE = Path("/Users/921623492/Abdoul Thesis/src/STREAMLINE")
ORIGINAL_DATA = SRC_STREMLINE / "data"
OUT = Path("/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output")
TRAIN_NAME = "SNP_merged_train"

CV_DIR = OUT / TRAIN_NAME / "CVDatasets"
MODEL_DIR = OUT / TRAIN_NAME / "models" / "pickledModels"


In [3]:
snp_train = ORIGINAL_DATA / "SNPData"
snp_test = ORIGINAL_DATA / "SNPRepData"
snp_train_data = snp_train / "SNP_merged_train.csv"
snp_test_data = snp_test / "SNP_merged_test.csv" # untouched 

snp_train_data = pd.read_csv(snp_train_data)
snp_test_data = pd.read_csv(snp_test_data)


In [4]:
snp_train_data.shape


(470, 56719)

In [5]:
snp_test_data.shape


(118, 56719)

### Step 1 : Streamline internal slpitting 
#### Cross-validation splits 
#### Exploratory 
#### Feature Selection
#### Model

In [6]:
# Cross-validation splits 
for fold in range(3):
    train_data = pd.read_csv(CV_DIR / f"SNP_merged_train_CV_{fold}_Train.csv")
    test_data = pd.read_csv(CV_DIR / f"SNP_merged_train_CV_{fold}_Test.csv")
    print(f"Fold {fold}: train = {train_data.shape}, test = {test_data.shape}")


Fold 0: train = (313, 5002), test = (157, 5002)
Fold 1: train = (313, 5002), test = (157, 5002)
Fold 2: train = (314, 5002), test = (156, 5002)


In [7]:
# Initial Folder 
data_processins_summary = pd.read_csv("/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/exploratory/DataProcessSummary.csv")


In [8]:
exploratory = OUT / TRAIN_NAME / "exploratory"
encoding_map = pd.read_csv(exploratory / "Numerical_Encoding_Map.csv")
encoding_map.head()


,Unnamed: 0,Category,Encoding
0,label,"['R', 'S']","[0, 1]"


In [9]:
# one hot encoding 
import pickle

with open(exploratory / "one_hot_feature.pickle", "rb") as f:
    one_hot_features = pickle.load(f)
type(one_hot_features), len(one_hot_features)


(list, 15698)

In [10]:
one_hot_features[:10]


['SNP_29_0',
 'SNP_29_1',
 'SNP_29_2',
 'SNP_42_0',
 'SNP_42_1',
 'SNP_42_2',
 'SNP_53_0',
 'SNP_53_1',
 'SNP_53_2',
 'SNP_67_0']

In [11]:
# count unique base SNPs | Number of origine SNP sites that were multi-allelic
base_snps = {name.rsplit("_", 1)[0] for name in one_hot_features}
len(base_snps)


5156

In [12]:
with open(exploratory / "ordinal_encoding.pickle", "rb") as f:
    ordinal_encoding = pickle.load(f)
type(ordinal_encoding), len(ordinal_encoding)


(pandas.core.frame.DataFrame, 1)

In [13]:
ordinal_encoding.head()


,Category,Encoding
label,"[R, S]","[0, 1]"


In [14]:
processed_categorical = pd.read_csv(exploratory / "processed_categorical_features.csv")
len(processed_categorical)


67259

In [15]:
with open(exploratory / "post_processed_features.pickle", "rb") as f:
    post_processed_features = pickle.load(f)
type(post_processed_features), len(post_processed_features)


(list, 67261)

In [16]:
post_processed_features[:10]


['sample_id',
 'SNP_0',
 'SNP_1',
 'SNP_2',
 'SNP_3',
 'SNP_4',
 'SNP_5',
 'SNP_6',
 'SNP_7',
 'SNP_8']

In [17]:
data_processins_summary.head()


,Unnamed: 0,Instances,Total Features,Categorical Features,Quantitative Features,Missing Values,Missing Percent,Class 0,Class 1
0,Original,470.0,56717.0,56717.0,0.0,0.0,0.0,250.0,220.0
1,C1,470.0,56717.0,56717.0,0.0,0.0,0.0,220.0,250.0
2,E1,470.0,56717.0,56717.0,0.0,0.0,0.0,220.0,250.0
3,C2,470.0,56717.0,56717.0,0.0,0.0,0.0,220.0,250.0
4,C3,470.0,56717.0,56717.0,0.0,0.0,0.0,220.0,250.0


In [18]:
# Univariate analysis 
# How does each feature relate to the label individually?
univariate_analysis = OUT / "SNP_merged_train" / "exploratory" / "Univariate_analyses"
df_univariate_analysis = pd.read_csv("/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/exploratory/univariate_analyses/Univariate_Significance.csv")


In [19]:
df_univariate_analysis.head()


,Feature,p-value,Test-statistic,Test-name
0,SNP_0,0.185047,1.756614,Chi Square Test
1,SNP_1,0.848844,0.036326,Chi Square Test
2,SNP_2,0.006481,7.411445,Chi Square Test
3,SNP_3,0.000589,11.810124,Chi Square Test
4,SNP_4,0.389869,0.739352,Chi Square Test


In [20]:
df_univariate_analysis.shape


(67259, 4)

In [21]:
# Feature selection 

import pandas as pd

mi0 = pd.read_csv(
    OUT / TRAIN_NAME / "feature_selection" / "mutual_information" /
    "mutual_information_scores_cv_0.csv"
)

mi0.head()


,Sorted mutual_information Scores
SNP_17651,0.548810
SNP_12955_1,0.527269
SNP_12955_0,0.493377
SNP_10068,0.267196
SNP_6421,0.260930


In [22]:
summary = pd.read_csv(
    OUT / TRAIN_NAME / "feature_selection" /
    "InformativeFeatureSummary.csv"
)

summary.head()


,CV_Partition,Informative,Uninformative
0,0,41828,25431
1,1,42679,24580
2,2,42026,25233


### Models 


In [ ]:
MODEL_DIR = OUT / TRAIN_NAME / "models" / "pickledModels"


In [24]:
snp_test_data.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56708,SNP_56709,SNP_56710,SNP_56711,SNP_56712,SNP_56713,SNP_56714,SNP_56715,SNP_56716,label
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,R
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,0,0,1,1,0,1,1,1,1,R
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,R
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,R
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,0,0,1,1,0,1,1,1,1,R


In [27]:
# encode label 
snp_test_data['label'] = snp_test_data['label'].astype(str).str.strip().map({"R": 0, "S": 1})


In [28]:
snp_test_data.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56708,SNP_56709,SNP_56710,SNP_56711,SNP_56712,SNP_56713,SNP_56714,SNP_56715,SNP_56716,label
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,0
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,0,0,1,1,0,1,1,1,1,0
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,0
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,0,1,1,1,0,1,1,0,2,0
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,0,0,1,1,0,1,1,1,1,0


In [ ]:
snp_test_processed = pd.read_csv('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/replication/SNP_merged_test/SNP_merged_test_Processed.csv')
snp_test_processed.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56699_0,SNP_56699_1,SNP_56699_2,SNP_56700_0,SNP_56700_1,SNP_56700_2,SNP_56700_3,SNP_56716_0,SNP_56716_1,SNP_56716_2
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,False,False,True,False,False,True,0,0,0,0
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,False,True,False,False,True,False,0,0,0,0
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,False,True,False,False,True,False,0,0,0,0


In [29]:
## merging label to processed test data 
snp_test_processed = pd.merge(snp_test_processed, snp_test_data[['sample_id', 'label']], on='sample_id', how='left')
snp_test_processed.head()


,sample_id,SNP_0,SNP_1,SNP_2,SNP_3,SNP_4,SNP_5,SNP_6,SNP_7,SNP_8,...,SNP_56699_1,SNP_56699_2,SNP_56700_0,SNP_56700_1,SNP_56700_2,SNP_56700_3,SNP_56716_0,SNP_56716_1,SNP_56716_2,label_y
0,ERR1218582,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
1,ERR1218605,0,0,0,1,0,0,1,0,0,...,False,True,False,False,True,0,0,0,0,0
2,ERR1218607,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
3,ERR1218623,0,0,0,1,1,0,1,1,1,...,True,False,False,True,False,0,0,0,0,0
4,ERR1218646,0,1,1,1,0,0,1,0,0,...,True,False,False,True,False,0,0,0,0,0


In [30]:
cv_paths = sorted(Path(CV_DIR).glob(f"{TRAIN_NAME}_CV_*_Train.csv"))
for cv_path in cv_paths:
    cols = pd.read_csv(cv_path, nrows=0).columns
    feats = [c for c in cols if c not in ["label", "sample_id"]]
    print(f"{cv_path.name}: {len(feats)} features, hash={hash(tuple(feats))}")


SNP_merged_train_CV_0_Train.csv: 5000 features, hash=2111208474750923444
SNP_merged_train_CV_1_Train.csv: 5000 features, hash=5620382050294848421
SNP_merged_train_CV_2_Train.csv: 5000 features, hash=2034895008020827941


In [32]:
def build_test_matrix_for_fold(test_data_processed, cv_train_csv, label_col="label_y"):
    """
    Align processed test data to the feature space of a specific CV fold.

    test_data_processed: processed test dataframe (has sample_id, label_raw, all 67k features)
    cv_train_csv: path to e.g. SNP_merged_train_CV_0_Train.csv
    label_col:    column in test_data_processed with numeric labels (0/1)
    """
    # Get fold-specific feature list
    cv_cols = pd.read_csv(cv_train_csv, nrows=0).columns
    feat_cols = [c for c in cv_cols if c not in ["label", "sample_id"]]

    # Check all features are present in test
    missing = [c for c in feat_cols if c not in test_data_processed.columns]
    if missing:
        raise ValueError(
            f"{cv_train_csv} expects {len(feat_cols)} features, "
            f"but {len(missing)} are missing in test_data_processed. "
            f"First few missing: {missing[:20]}"
        )

    # Build X, y, ids
    X = test_data_processed[feat_cols].to_numpy()
    y = test_data_processed[label_col].astype(int).to_numpy()
    ids = test_data_processed["sample_id"].astype(str).to_numpy()

    return X, y, ids, feat_cols


In [ ]:
cv_paths = sorted(Path(CV_DIR).glob(f"{TRAIN_NAME}_CV_*_Train.csv"))

for cv_path in cv_paths:
    print(f"\n=== {cv_path.name} ===")
    X_fold, y_fold, ids_fold, feat_cols_fold = build_test_matrix_for_fold(snp_test_processed, cv_path)
    print("X shape:", X_fold.shape, "y shape:", y_fold.shape)  



=== SNP_merged_train_CV_0_Train.csv ===
X shape: (118, 5000) y shape: (118,)

=== SNP_merged_train_CV_1_Train.csv ===
X shape: (118, 5000) y shape: (118,)

=== SNP_merged_train_CV_2_Train.csv ===
X shape: (118, 5000) y shape: (118,)


### Model Loading 


In [ ]:
MODEL_DIR
list(MODEL_DIR.iterdir())


[PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/LR_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/XGB_0.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_1.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/RF_0.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/KNN_2.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged_train/models/pickledModels/KNN_1.pickle'),
 PosixPath('/Users/921623492/Abdoul Thesis/results/snp/2025-12-24/snp_output/SNP_merged

### Step 1: Load all LR models and extract parameters


In [49]:
rows = []

for model_path in sorted(MODEL_DIR.glob("LR_*.pickle")):
    model = joblib.load(model_path)
    fold = int(model_path.stem.split("_")[-1])

    params = model.get_params()
    params["fold"] = fold

    rows.append(params)

df_lr_params = pd.DataFrame(rows).set_index("fold")
df_lr_params



,C,class_weight,dual,fit_intercept,intercept_scaling,l1_ratio,max_iter,multi_class,n_jobs,penalty,random_state,solver,tol,verbose,warm_start
fold,,,,,,,,,,,,,,,
0,2090.900813,None,False,True,1,None,730,auto,None,l2,42,newton-cg,0.0001,0,False
1,97739.384669,None,False,True,1,None,46,auto,None,l2,42,lbfgs,0.0001,0,False
2,0.196705,balanced,False,True,1,None,97,auto,None,l1,42,liblinear,0.0001,0,False


In [50]:
rows = []

for model_path in sorted(MODEL_DIR.glob("RF_*.pickle")):
    model = joblib.load(model_path)
    fold = int(model_path.stem.split("_")[-1])

    params = model.get_params()
    params["fold"] = fold

    rows.append(params)

df_RF_params = pd.DataFrame(rows).set_index("fold")
df_RF_params


,bootstrap,ccp_alpha,class_weight,criterion,max_depth,max_features,max_leaf_nodes,max_samples,min_impurity_decrease,min_samples_leaf,min_samples_split,min_weight_fraction_leaf,n_estimators,n_jobs,oob_score,random_state,verbose,warm_start
fold,,,,,,,,,,,,,,,,,,
0,True,0.0,None,gini,18,auto,None,None,0.0,8,9,0.0,381,None,False,42,0,False
1,True,0.0,None,gini,18,auto,None,None,0.0,8,9,0.0,381,None,False,42,0,False
2,True,0.0,None,gini,18,auto,None,None,0.0,8,9,0.0,381,None,False,42,0,False


In [51]:
rows = []

for model_path in sorted(MODEL_DIR.glob("XGB_*.pickle")):
    model = joblib.load(model_path)
    fold = int(model_path.stem.split("_")[-1])

    params = model.get_params()
    params["fold"] = fold

    rows.append(params)

df_xgb_params = pd.DataFrame(rows).set_index("fold")
df_xgb_params


,objective,base_score,booster,callbacks,colsample_bylevel,colsample_bynode,colsample_bytree,device,early_stopping_rounds,enable_categorical,...,scale_pos_weight,subsample,tree_method,validate_parameters,verbosity,alpha,eta,min_samples_split,min_samples_leaf,nthread
fold,,,,,,,,,,,,,,,,,,,,,
0,binary:logistic,None,gbtree,None,None,None,0.849198,None,None,False,...,1.0,0.510292,None,None,0,0.40338,0.007177,31,36,1
1,binary:logistic,None,gbtree,None,None,None,0.849198,None,None,False,...,1.0,0.510292,None,None,0,0.40338,0.007177,31,36,1
2,binary:logistic,None,gbtree,None,None,None,0.849198,None,None,False,...,1.0,0.510292,None,None,0,0.40338,0.007177,31,36,1


In [52]:
rows = []

for model_path in sorted(MODEL_DIR.glob("KNN_*.pickle")):
    model = joblib.load(model_path)
    fold = int(model_path.stem.split("_")[-1])

    params = model.get_params()
    params["fold"] = fold

    rows.append(params)

df_knn_params = pd.DataFrame(rows).set_index("fold")
df_knn_params


,algorithm,leaf_size,metric,metric_params,n_jobs,n_neighbors,p,weights
fold,,,,,,,,
0,auto,30,minkowski,None,None,4,1,distance
1,auto,30,euclidean,None,None,42,3,distance
2,auto,30,minkowski,None,None,4,1,distance


In [53]:
rows = []

for model_path in sorted(MODEL_DIR.glob("DT_*.pickle")):
    model = joblib.load(model_path)
    fold = int(model_path.stem.split("_")[-1])

    params = model.get_params()
    params["fold"] = fold

    rows.append(params)

df_dt_params = pd.DataFrame(rows).set_index("fold")
df_dt_params


,ccp_alpha,class_weight,criterion,max_depth,max_features,max_leaf_nodes,min_impurity_decrease,min_samples_leaf,min_samples_split,min_weight_fraction_leaf,random_state,splitter
fold,,,,,,,,,,,,
0,0.0,balanced,gini,29,None,None,0.0,30,45,0.0,42,best
1,0.0,balanced,gini,4,None,None,0.0,2,26,0.0,42,best
2,0.0,balanced,gini,29,None,None,0.0,30,45,0.0,42,best
